# Olist delivery-demand hotspots

**Business question:** Which areas have enough historical customer orders to investigate as possible pickup-point or delivery-hub service zones? This is *unsupervised learning*: there are no correct zone labels to predict. We will find dense areas, not an exact building address.

**Unit of analysis:** start with **one row per order**. Then count orders at each customer ZIP-code prefix, and use those counts to give busy prefixes more weight during clustering. We need only the `orders`, `customers` and `geolocation` tables. Joining order items or payments would duplicate orders and give incorrect demand counts.

**Workflow:** load -> check quality -> join safely -> explore orders -> count by ZIP prefix -> plot demand -> train DBSCAN -> inspect and map hotspots. The dataset is historical (mostly 2016-2018); these maps are examples for learning, not current site recommendations.

## 1. Import libraries and locate the dataset

Run root `scripts/bootstrap.py` once and open this notebook from within the cloned repository. The static charts need pandas, NumPy, Matplotlib and scikit-learn; the bootstrap also installs Folium for interactive maps. Map background tiles need internet access.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.cluster import DBSCAN

try:
    import folium
    from folium.plugins import HeatMap
except ImportError:
    folium = None
    print('Folium is optional. Static geographic plots will still work.')

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'datasets' / 'olist' / 'Ecommerce Dataset.zip').is_file()), None)
assert ROOT is not None, 'Open the notebook from inside the cloned course repository.'
DATA = ROOT / 'datasets' / 'olist' / 'Ecommerce Dataset'
needed = ['olist_orders_dataset.csv', 'olist_customers_dataset.csv',
          'olist_geolocation_dataset.csv']
assert all((DATA / name).is_file() for name in needed), (
    f'Missing files in {DATA.resolve()}; run scripts/bootstrap.py from the repo root first.'
)
print('Data folder:', DATA.resolve())

## 2. Load the three tables

Orders tell us **how many purchases happened and when**. Customers provide the ZIP-code prefix and state. Geolocation provides approximate coordinates for a prefix. Read ZIP prefixes as text so a leading zero is not lost. We intentionally skip unrelated columns to keep the notebook quick.

In [ ]:
orders = pd.read_csv(
    DATA / 'olist_orders_dataset.csv',
    usecols=['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp'],
    parse_dates=['order_purchase_timestamp'],
)
customers = pd.read_csv(
    DATA / 'olist_customers_dataset.csv',
    usecols=['customer_id', 'customer_zip_code_prefix', 'customer_state'],
    dtype={'customer_zip_code_prefix': 'string'},
)
geo = pd.read_csv(
    DATA / 'olist_geolocation_dataset.csv',
    usecols=['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng'],
    dtype={'geolocation_zip_code_prefix': 'string'},
)
print('Orders:', orders.shape, '| Customers:', customers.shape, '| Geolocation:', geo.shape)
display(orders.head(3))
display(customers.head(3))
display(geo.head(3))

## 3. Check keys, dates and order status

A correct demand count needs every order to appear **once**. Check that `order_id` and `customer_id` are unique in their own tables. Look at the time range and statuses. We count all placed orders as historical demand, including a small number later canceled; for a fulfilled-only study you could filter to `delivered` and explain that different question.

In [ ]:
assert orders.order_id.is_unique
assert customers.customer_id.is_unique
print('Unique orders:', orders.order_id.nunique())
print('Purchase range:', orders.order_purchase_timestamp.min(),
      'to', orders.order_purchase_timestamp.max())
print('Orders without customer ID:', orders.customer_id.isna().sum())
print('Customer ZIP prefixes missing:', customers.customer_zip_code_prefix.isna().sum())
display(orders.order_status.value_counts().to_frame('orders'))
print('Geolocation rows:', len(geo), '| distinct ZIP prefixes:',
      geo.geolocation_zip_code_prefix.nunique())

## 4. Turn repeated geolocation rows into a ZIP-prefix lookup

One ZIP prefix can have **many** coordinate rows. A direct join would multiply orders and corrupt the counts. First remove coordinates far outside an approximate Brazil rectangle and take the **median** latitude and longitude per prefix. The rectangle is a simple data check, not a precise border; the result is a ZIP-prefix approximation, not a customer's address.

In [ ]:
inside_brazil_box = (geo.geolocation_lat.between(-34, 6) &
                     geo.geolocation_lng.between(-74, -34))
print('Coordinates outside approximate bounds:', (~inside_brazil_box).sum())
zip_coords = geo.loc[inside_brazil_box].groupby('geolocation_zip_code_prefix')[
    ['geolocation_lat', 'geolocation_lng']
].median()
assert zip_coords.index.is_unique
print('One coordinate pair per prefix:', len(zip_coords))
display(zip_coords.head())

## 5. Build the order-level location dataset

Join each order to its customer, then look up that customer's ZIP-prefix coordinates. Check the row count **after each join**. Keep a copy of all orders, and use only geocoded orders for spatial analysis. Report how many orders cannot be shown on a map instead of quietly forgetting them.

In [ ]:
order_locations = orders.merge(customers, on='customer_id',
                               how='left', validate='many_to_one')
assert len(order_locations) == len(orders)
order_locations = order_locations.join(
    zip_coords, on='customer_zip_code_prefix', validate='many_to_one')
assert len(order_locations) == len(orders)
geo_orders = order_locations.dropna(
    subset=['geolocation_lat', 'geolocation_lng', 'order_purchase_timestamp']
).copy()
print('Total orders:', len(orders))
print('Orders with usable location and date:', len(geo_orders))
print('Orders omitted from map:', len(orders) - len(geo_orders))
print('Mapped share: {:.1%}'.format(len(geo_orders) / len(orders)))
assert geo_orders.order_id.is_unique
display(geo_orders[['order_id', 'order_purchase_timestamp',
                    'customer_zip_code_prefix', 'customer_state',
                    'geolocation_lat', 'geolocation_lng']].head())

## 6. EDA: when and where do orders occur?

Before any model, count orders by purchase month and customer state. This tells us how much history we have and which states dominate. These are **raw counts**, not orders per resident; population differences matter when interpreting them.

In [ ]:
monthly_orders = geo_orders.groupby(
    geo_orders.order_purchase_timestamp.dt.to_period('M')
).size()
state_orders = geo_orders.customer_state.value_counts()
display(state_orders.head(10).to_frame('orders'))
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(monthly_orders.index.to_timestamp(), monthly_orders.values)
axes[0].set(title='Geocoded orders per purchase month', ylabel='Orders')
state_orders.head(10).sort_values().plot.barh(ax=axes[1], color='#397c9a')
axes[1].set(title='Ten states with the most orders', xlabel='Orders')
plt.tight_layout()
plt.show()

## 7. Feature engineering: count orders per ZIP prefix

Now make the dataset for clustering: **one row per customer ZIP-code prefix**. `order_count` is the number of order IDs there, not the number of geolocation records. Its coordinate is the representative point calculated earlier. A busy prefix should contribute more evidence of demand than a prefix with one order.

In [ ]:
zip_demand = geo_orders.groupby('customer_zip_code_prefix', as_index=False).agg(
    order_count=('order_id', 'size'),
    latitude=('geolocation_lat', 'first'),
    longitude=('geolocation_lng', 'first'),
    state=('customer_state', 'first'),
)
assert zip_demand.customer_zip_code_prefix.is_unique
assert zip_demand.order_count.sum() == len(geo_orders)
assert zip_demand[['latitude', 'longitude']].notna().all().all()
print('ZIP prefixes with orders:', len(zip_demand))
print('Orders recovered from prefix counts:', zip_demand.order_count.sum())
display(zip_demand.sort_values('order_count', ascending=False).head(10))

## 8. EDA: are orders evenly spread across ZIP prefixes?

Compare a typical prefix with the busiest prefixes. The histogram answers: how many prefixes have only a few orders, and how many have lots of orders? This is why treating every prefix as one equal customer would be misleading.

In [ ]:
display(zip_demand.order_count.describe(percentiles=[.5, .9, .99])
        .to_frame('orders_per_prefix'))
fig, ax = plt.subplots(figsize=(8, 4))
ax.hist(zip_demand.order_count, bins=40, color='#46789b')
ax.set(title='How many orders are in each ZIP prefix?',
       xlabel='Orders at a ZIP prefix', ylabel='Number of prefixes')
plt.show()

## 9. Show a demand heat-style plot (works offline)

Each hexagon combines nearby ZIP prefixes and **adds their order counts**. Darker means more historical orders. This is a geographic coordinate plot, not a road map; the interactive map later adds a basemap when map tiles can load. The color scale is logarithmic so smaller hotspots remain visible.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 8))
heat = ax.hexbin(zip_demand.longitude, zip_demand.latitude,
                 C=zip_demand.order_count, reduce_C_function=np.sum,
                 gridsize=55, mincnt=1, bins='log', cmap='YlOrRd')
fig.colorbar(heat, ax=ax, label='Orders per area (log color scale)')
ax.set(xlabel='Longitude', ylabel='Latitude',
       title='Historical customer-order density by location',
       xlim=(-74, -34), ylim=(-34, 6))
ax.set_aspect('equal')
plt.show()

## 10. Train a density-based clustering model

DBSCAN finds nearby, busy areas and leaves sparse areas as **noise** (`cluster = -1`). We give each ZIP prefix its `order_count` as a weight: **100 nearby orders** meet the density threshold even if spread over several prefixes. `10 km` defines each point's neighborhood; a whole connected cluster can be wider than 10 km. Haversine distance works with latitude/longitude in radians. These two settings are illustrative and can be changed.

In [ ]:
earth_radius_km = 6371.0088
radius_km = 10
nearby_orders_needed = 100
coordinates_rad = np.radians(zip_demand[['latitude', 'longitude']].to_numpy())
weights = zip_demand.order_count.to_numpy()
model = DBSCAN(eps=radius_km / earth_radius_km,
               min_samples=nearby_orders_needed,
               metric='haversine', algorithm='ball_tree')
zip_demand['cluster'] = model.fit(coordinates_rad, sample_weight=weights).labels_
print('Hotspot clusters:', zip_demand.loc[zip_demand.cluster.ge(0), 'cluster'].nunique())
print('ZIP prefixes outside clusters:', zip_demand.cluster.eq(-1).sum())
print('Orders in clustered ZIP prefixes:',
      zip_demand.loc[zip_demand.cluster.ge(0), 'order_count'].sum())

## 11. Turn cluster labels into an actionable summary

Cluster numbers are arbitrary IDs. Rank hotspots by **number of orders**, count the ZIP prefixes in each, and compute an order-weighted center for display. The center is a visual marker, **not** an optimal pickup-point address. Noise does not mean zero demand; it means the area did not meet our density setting.

In [ ]:
clustered = zip_demand[zip_demand.cluster.ge(0)].copy()
clustered['lat_times_orders'] = clustered.latitude * clustered.order_count
clustered['lon_times_orders'] = clustered.longitude * clustered.order_count
hotspots = clustered.groupby('cluster').agg(
    orders=('order_count', 'sum'),
    zip_prefixes=('order_count', 'size'),
    lat_total=('lat_times_orders', 'sum'),
    lon_total=('lon_times_orders', 'sum'),
)
hotspots['main_state'] = clustered.groupby('cluster').state.agg(
    lambda values: values.mode().iloc[0])
hotspots['center_lat'] = hotspots.lat_total / hotspots.orders
hotspots['center_lon'] = hotspots.lon_total / hotspots.orders
hotspots['share_of_mapped_orders_pct'] = (
    100 * hotspots.orders / len(geo_orders)).round(1)
hotspots = hotspots.drop(columns=['lat_total', 'lon_total'])
hotspots = hotspots.sort_values('orders', ascending=False).reset_index()
hotspots.insert(0, 'rank', np.arange(1, len(hotspots) + 1))
print('Share of mapped orders inside hotspots: {:.1%}'.format(
    hotspots.orders.sum() / len(geo_orders)))
display(hotspots.head(12).round(2))

## 12. Show which ZIP prefixes belong to the largest clusters

On this offline coordinate map, colored points belong to the eight busiest hotspots. Point size indicates orders at that prefix. Gray points are smaller clusters **or** sparse/noise areas. The X marker is only the hotspot's order-weighted center. This view shows the actual cluster spread, which a single center marker cannot show.

In [ ]:
top_eight = hotspots.head(8)
other = zip_demand[~zip_demand.cluster.isin(top_eight.cluster)]
fig, ax = plt.subplots(figsize=(11, 9))
ax.scatter(other.longitude, other.latitude, s=3, color='#b4bec3',
           alpha=.4, label='Other / sparse prefixes', rasterized=True)
colors = plt.get_cmap('tab10')
for index, zone in enumerate(top_eight.itertuples(index=False)):
    points = zip_demand[zip_demand.cluster.eq(zone.cluster)]
    color = colors(index)
    ax.scatter(points.longitude, points.latitude,
               s=np.clip(points.order_count * 1.5, 4, 70),
               color=color, alpha=.55, rasterized=True,
               label=f'#{zone.rank}: {zone.main_state} ({zone.orders:,} orders)')
    ax.scatter(zone.center_lon, zone.center_lat, marker='x',
               color=color, s=65, linewidths=2)
ax.set(xlabel='Longitude', ylabel='Latitude',
       title='Eight largest order-density clusters',
       xlim=(-74, -34), ylim=(-34, 6))
ax.set_aspect('equal')
ax.legend(loc='lower left', fontsize=7, ncol=2)
plt.show()

## 13. Explore an interactive map

The heat layer uses ZIP-prefix order counts; colored circles mark the **centers** of the ten largest discovered hotspots. Turn layers on or off using the control at the top right. The colored circles do not describe a cluster's exact boundary; use the previous plot for membership. Folium requires an internet connection to load the background map tiles. Without Folium, use the two static plots above.

In [ ]:
if folium is None:
    print('Interactive map skipped. Install Folium to display it.')
else:
    map_view = folium.Map(location=[-15, -51], zoom_start=4,
                          tiles='CartoDB positron', control_scale=True)
    HeatMap(zip_demand[['latitude', 'longitude', 'order_count']].to_numpy().tolist(),
            radius=14, blur=17, name='Order density').add_to(map_view)
    markers = folium.FeatureGroup(name='Ten biggest hotspot centers')
    map_colors = ['red', 'blue', 'green', 'purple', 'orange',
                  'darkred', 'cadetblue', 'darkgreen', 'pink', 'black']
    for index, zone in enumerate(hotspots.head(10).itertuples(index=False)):
        folium.CircleMarker(
            location=[zone.center_lat, zone.center_lon],
            radius=8, color=map_colors[index], fill=True, fill_opacity=.9,
            tooltip=f'Hotspot #{zone.rank}: {zone.orders:,} orders',
            popup=f'Rank {zone.rank} | State: {zone.main_state} | '
                  f'{zone.orders:,} orders | {zone.zip_prefixes:,} ZIP prefixes',
        ).add_to(markers)
    markers.add_to(map_view)
    folium.LayerControl().add_to(map_view)
    display(map_view)

## 14. Check whether the result depends on our radius choice

Unsupervised models have no accuracy score against true hotspot labels here. As a basic **sensitivity check**, fit again with 7 km and 15 km. Report how many groups and orders each setting covers. Wider neighborhoods may connect several towns into one large zone; choose settings based on the business's practical service-area size, not just the largest coverage percentage.

In [ ]:
comparisons = []
for km in [7, 10, 15]:
    labels = DBSCAN(eps=km / earth_radius_km,
                    min_samples=nearby_orders_needed,
                    metric='haversine', algorithm='ball_tree').fit(
                        coordinates_rad, sample_weight=weights).labels_
    grouped_orders = zip_demand.loc[labels >= 0].groupby(labels[labels >= 0])[
        'order_count'].sum()
    comparisons.append({
        'radius_km': km, 'hotspots': len(grouped_orders),
        'orders_in_hotspots': int(grouped_orders.sum()),
        'share_of_mapped_orders': round(grouped_orders.sum() / len(geo_orders), 3),
        'largest_hotspot_orders': int(grouped_orders.max()),
    })
display(pd.DataFrame(comparisons))

## 15. What the model can and cannot decide

**Useful output:** a ranked list of dense historical demand areas, a map, and the order count behind each area. These can guide *which areas to investigate first* for a pickup-point pilot. A simple ZIP-count map is the baseline; DBSCAN adds connected service-area groups across ZIP-prefix boundaries.

**Not solved:** the best actual street address, road travel time, hub capacity, site rent, local population, missed demand, or whether these 2016-2018 patterns still hold today. A cluster can be wide; its center is not a recommended site. For a real rollout, combine current orders, population, transport, capacity and site costs, and confirm demand is stable in later months.

**Practice:** change `nearby_orders_needed` or `radius_km`, rerun from Step 10 and explain the changes. Or rerun Steps 5-14 with only `delivered` orders to study fulfilled demand instead of all placed orders.